# Exportar best.pt a ONNX (para la prueba en el celular)

1. Subí `best.pt` al panel de archivos de Colab, o usá la ruta de Drive donde ya está.
2. Ajustá las variables de la primera celda y corré las celdas en orden.
3. Los `.onnx` se descargan a tu PC: copialos a `ServidorIA/ProcesamientoCelular/navegador/`.

In [ ]:
# ---- Variables: cambiá solo esto ----
PESOS = "/content/best.pt"      # donde está el modelo (puede ser una ruta de Drive)
SALIDA = "/content"             # carpeta donde quedan los .onnx
TAMANOS = [640, 960]            # tamaños de entrada a exportar
OPSET = 12
# --------------------------------------

import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'   # evita el error de torch.load

!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -q onnx ultralytics

assert os.path.exists(PESOS), f"No encuentro {PESOS}: subí best.pt o corregí la ruta"
print("OK, modelo encontrado:", PESOS, f"({os.path.getsize(PESOS)/1e6:.1f} MB)")

In [ ]:
import onnx

for t in TAMANOS:
    !python export.py --weights {PESOS} --include onnx --imgsz {t} --opset {OPSET}
    ruta = PESOS.replace(".pt", ".onnx")
    modelo = onnx.load(ruta)    # lee también el archivo de datos externo, si existe
    # todo en un solo archivo: el navegador no puede buscar un .onnx.data aparte
    onnx.save_model(modelo, f"{SALIDA}/best_{t}.onnx", save_as_external_data=False)

!ls -la {SALIDA}/*.onnx   # cada uno debería pesar ~28 MB

In [ ]:
from google.colab import files
for t in TAMANOS:
    files.download(f"{SALIDA}/best_{t}.onnx")